# MobileNetV4 UIB blocks in a ConvNeXt network, trained on CIFAR-10 (BatchNorm vs LayerNorm)

This notebook goes with the report in `README.md`. It:
1. builds the ConvNeXt-UIB model (the UIB block comes from `timm`),
2. trains it with AdamW (set `RUN_TRAINING = True`; on a Colab GPU one 30-epoch run takes a few minutes),
3. plots Loss / Accuracy curves and the BatchNorm vs LayerNorm experiments.

The saved results in `results/` were produced on a 4-core CPU with `run_experiments.sh`.

In [ ]:
# On Colab:
# !git clone -b claude/nice-knuth-ot17ij https://github.com/alirezasa/MobileNetV4.git && %cd MobileNetV4
# !pip install -q timm
import json, torch, timm
from src.models import create_model, count_params, UIB_VARIANTS
print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())
RUN_TRAINING = False   # True -> retrain every run (writes results/*.json)
DATA = './data'        # CIFAR-10 is downloaded here by torchvision

## 1. The model
A ConvNeXt layout (patchify stem → 4 stages separated by `norm + 2×2/2 conv` downsampling → GAP → norm → linear).
Every block is `timm.models._efficientnet_blocks.UniversalInvertedResidual` (UIB) in its **ConvNeXt-like** variant:
`DW 7×7 → PW expand ×4 (+GELU) → PW project → LayerScale → DropPath → + residual`.
The normalisation layer (`norm='bn'|'ln'|'gn1'|'none'`) is passed to every UIB and to the stem/downsampling layers.

In [ ]:
print(UIB_VARIANTS)
for norm in ['bn', 'ln']:
    m = create_model('tiny', norm=norm)
    print(f'{norm}: {count_params(m)/1e6:.3f} M params')
m = create_model('tiny', norm='ln')
print(m.stages[1])

In [ ]:
from timm.utils.model import get_state_dict  # noqa  (timm utility, just to show the package is used)
x = torch.randn(2, 3, 32, 32)
for norm in ['bn', 'ln', 'gn1', 'none']:
    print(norm, create_model('tiny', norm=norm)(x).shape)

## 2. Training
`train.py` uses timm's `create_optimizer_v2('adamw')` (no weight decay on norm/bias parameters), `CosineLRScheduler`
with linear warmup, `LabelSmoothingCrossEntropy`, and `accuracy` / `AverageMeter`.

In [ ]:
if RUN_TRAINING:
    !python train.py --data {DATA} --norm bn --epochs 30 --save-ckpt checkpoints/main_bn.pt --out results/main_bn.json
    !python train.py --data {DATA} --norm ln --epochs 30 --save-ckpt checkpoints/main_ln.pt --out results/main_ln.json
    for bs in [4, 16, 128]:
        lr = 2e-3 * (bs / 128) ** 0.5
        for norm in ['bn', 'ln']:
            !python train.py --data {DATA} --norm {norm} --batch-size {bs} --lr {lr} --epochs 10 --warmup-epochs 1 --train-subset 10000 --eval-train-mode --out results/bs_{norm}_{bs}.json
    !python train.py --data {DATA} --norm none --epochs 10 --warmup-epochs 1 --train-subset 10000 --out results/bs_none_128.json

## 3. Loss and accuracy curves (experiment 1)

In [ ]:
!python plot_results.py > /dev/null
from IPython.display import Image, Markdown, display
display(Image('results/figures/main_curves.png'))
for n in ['bn', 'ln']:
    h = json.load(open(f'results/main_{n}.json'))
    print(n, 'epoch | train_loss | train_acc | test_loss | test_acc')
    for e in h['epochs']:
        if e['epoch'] % 5 == 0 or e['epoch'] == 1:
            print(f"   {e['epoch']:5d} | {e['train_loss']:.3f} | {e['train_acc']:6.2f} | {e['test_loss']:.3f} | {e['test_acc']:6.2f}")

In [ ]:
display(Image('results/figures/lr_schedule.png'))

## 4. BatchNorm vs LayerNorm (experiment 2: batch size)

In [ ]:
display(Image('results/figures/batch_size.png'))
display(Markdown(open('results/summary.md').read()))

## 5. BatchNorm vs LayerNorm: analysis without training (`bn_vs_ln_analysis.py`)

In [ ]:
if RUN_TRAINING:
    !python bn_vs_ln_analysis.py --data {DATA}
for f in ['norm_axes', 'batch_dependence', 'bn_test_batch_stats']:
    display(Image(f'results/figures/{f}.png'))
print(json.dumps(json.load(open('results/bn_vs_ln_analysis.json')), indent=1))